## Step 1: Load the train/val/test splits we saved in notebook 03


In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

# The text is already cleaned from notebook 03, but just in case of NaNs after loading:
train_df["text"] = train_df["text"].fillna("")
val_df["text"] = val_df["text"].fillna("")
test_df["text"] = test_df["text"].fillna("")

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

Train: (17244, 2)  Val: (3695, 2)  Test: (3696, 2)


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_df["text"])   # fit + transform on train
X_val = vectorizer.transform(val_df["text"])            # transform only (reuse train's vocabulary)
X_test = vectorizer.transform(test_df["text"])          # transform only

y_train = train_df["label"]
y_val = val_df["label"]
y_test = test_df["label"]

print("TF-IDF matrix shape (train):", X_train.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape (train): (17244, 4582)
Vocabulary size: 4582


## Step 3: Train the Linear SVM

`class_weight="balanced"` tells the model to pay extra attention to smaller
categories (like CANCEL) so it doesn't just favor the biggest category (ACCOUNT).


In [3]:
from sklearn.svm import LinearSVC

svm = LinearSVC(class_weight="balanced", max_iter=5000)
svm.fit(X_train, y_train)

print("SVM training complete")

SVM training complete


## Step 4: Train Logistic Regression (for comparison)


In [4]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight="balanced", max_iter=2000, n_jobs=-1)
logreg.fit(X_train, y_train)

print("Logistic Regression training complete")

c:\For ME\ticket-classisfication-system\ml-service\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression training complete


In [5]:
from sklearn.metrics import f1_score

svm_val_pred = svm.predict(X_val)
logreg_val_pred = logreg.predict(X_val)

svm_f1 = f1_score(y_val, svm_val_pred, average="macro")
logreg_f1 = f1_score(y_val, logreg_val_pred, average="macro")

print(f"Linear SVM        validation macro-F1: {svm_f1:.4f}")
print(f"Logistic Regr.    validation macro-F1: {logreg_f1:.4f}")

best_model, best_name = (svm, "LinearSVC") if svm_f1 >= logreg_f1 else (logreg, "LogisticRegression")
print(f"\nSelected best model: {best_name}")

Linear SVM        validation macro-F1: 0.9972
Logistic Regr.    validation macro-F1: 0.9959

Selected best model: LinearSVC


In [6]:
from sklearn.metrics import classification_report

test_pred = best_model.predict(X_test)
print(classification_report(y_test, test_pred))

              precision    recall  f1-score   support

     ACCOUNT       1.00      1.00      1.00       817
      CANCEL       1.00      1.00      1.00       142
     CONTACT       1.00      1.00      1.00       300
    DELIVERY       1.00      1.00      1.00       249
    FEEDBACK       1.00      1.00      1.00       300
     INVOICE       1.00      0.99      1.00       274
       ORDER       0.99      0.99      0.99       475
     PAYMENT       1.00      0.99      0.99       300
      REFUND       1.00      0.99      1.00       394
    SHIPPING       0.99      1.00      1.00       295
SUBSCRIPTION       1.00      1.00      1.00       150

    accuracy                           1.00      3696
   macro avg       1.00      1.00      1.00      3696
weighted avg       1.00      1.00      1.00      3696

